# **Smart Taxi Demand Prediction and Recommendation System**

**Import Libraries**

In [1]:
# Imports
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# Visualization (optional later)
import matplotlib.pyplot as plt
import seaborn as sns

**Mount Google Drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Read Multiple CSV Files (Chunk Processing)**

In [3]:
folder_path = "/content/drive/MyDrive/NYC_DATA"

files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

files

['yellow_tripdata_2015-01.csv',
 'yellow_tripdata_2016-01.csv',
 'yellow_tripdata_2016-02.csv',
 'yellow_tripdata_2016-03.csv']

**Setup Output File**

In [4]:
import os

output_file = "/content/cleaned_taxi.csv"

# Delete old file if exists
if os.path.exists(output_file):
    os.remove(output_file)

**Define Required Columns**

In [5]:
cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "trip_distance"
]

**Cleaning Function**

In [6]:
def clean_chunk(df):

    # Drop nulls
    df = df.dropna()

    # Convert numeric safely
    df["trip_distance"] = pd.to_numeric(df["trip_distance"], errors="coerce")

    # Remove invalid values
    df = df[df["trip_distance"] > 0]

    df = df[
        (df["pickup_longitude"] != 0) &
        (df["pickup_latitude"] != 0) &
        (df["dropoff_longitude"] != 0) &
        (df["dropoff_latitude"] != 0)
    ]

    return df

**Combine & Process All Files**

In [7]:
import pandas as pd
from tqdm import tqdm

first_chunk = True

for file in files:
    print(f"\nProcessing {file}")

    path = os.path.join(folder_path, file)

    try:
        for chunk in pd.read_csv(
            path,
            usecols=cols,
            chunksize=100000,
            low_memory=False
        ):

            # Convert datetime safely
            chunk["tpep_pickup_datetime"] = pd.to_datetime(
                chunk["tpep_pickup_datetime"], errors="coerce"
            )

            chunk["tpep_dropoff_datetime"] = pd.to_datetime(
                chunk["tpep_dropoff_datetime"], errors="coerce"
            )

            # Drop invalid datetime rows
            chunk = chunk.dropna(subset=[
                "tpep_pickup_datetime",
                "tpep_dropoff_datetime"
            ])

            # Clean data
            chunk = clean_chunk(chunk)

            # Skip empty chunks
            if chunk.empty:
                continue

            # Write to CSV
            chunk.to_csv(
                output_file,
                mode='a',
                header=first_chunk,
                index=False
            )

            first_chunk = False


            del chunk

    except Exception as e:
        print(f"Error in file {file}: {e}")
        continue

print("\n All files processed successfully!")


Processing yellow_tripdata_2015-01.csv

Processing yellow_tripdata_2016-01.csv

Processing yellow_tripdata_2016-02.csv

Processing yellow_tripdata_2016-03.csv

 All files processed successfully!


In [8]:
df = pd.read_csv(output_file, nrows=500000)

print(df.shape)
df.head()

(500000, 7)


,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
0,2015-01-15 19:05:39,2015-01-15 19:23:42,1.59,-73.993896,40.750111,-73.974785,40.750618
1,2015-01-10 20:33:38,2015-01-10 20:53:28,3.30,-74.001648,40.724243,-73.994415,40.759109
2,2015-01-10 20:33:38,2015-01-10 20:43:41,1.80,-73.963341,40.802788,-73.951820,40.824413
3,2015-01-10 20:33:39,2015-01-10 20:35:31,0.50,-74.009087,40.713818,-74.004326,40.719986
4,2015-01-10 20:33:39,2015-01-10 20:52:58,3.00,-73.971176,40.762428,-74.004181,40.742653


In [9]:
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

**Feature Engineering**

In [10]:
# Trip duration
df["trip_duration"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds()

# Remove invalid duration
df = df[df["trip_duration"] > 0]

# Time features
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_day"] = df["tpep_pickup_datetime"].dt.dayofweek
df["pickup_month"] = df["tpep_pickup_datetime"].dt.month

# Speed
df["speed"] = df["trip_distance"] / (df["trip_duration"] / 3600)

**Traffic Congestion Feature**

In [11]:
# Avoid division issues
df = df[df["speed"] > 0]

# Congestion (inverse of speed)
df["congestion"] = 1 / df["speed"]

# Normalize congestion
df["congestion"] = (
    (df["congestion"] - df["congestion"].min()) /
    (df["congestion"].max() - df["congestion"].min())
)

**Zone Creation using K-Means**

In [12]:
from sklearn.cluster import KMeans

# Use pickup coordinates
coords = df[["pickup_latitude", "pickup_longitude"]]

# Take sample (important for speed)
sample = coords.sample(n=50000, random_state=42)

kmeans = KMeans(n_clusters=20, random_state=42)
kmeans.fit(sample)

# Assign zones
df["Zone_ID"] = kmeans.predict(coords)

**Demand Aggregation**

In [13]:
demand_df = df.groupby(["Zone_ID", "pickup_hour"]).size().reset_index(name="demand")

demand_df.head()

,Zone_ID,pickup_hour,demand
0,0,0,1647
1,0,1,956
2,0,2,724
3,0,3,514
4,0,4,429


**Taxi Availability**

In [14]:
availability = df.groupby(["Zone_ID", "pickup_hour"]).size().reset_index(name="available_taxis")

availability.head()

,Zone_ID,pickup_hour,available_taxis
0,0,0,1647
1,0,1,956
2,0,2,724
3,0,3,514
4,0,4,429


**Merge Demand + Availability + Congestion**

In [15]:
merged = demand_df.merge(availability, on=["Zone_ID", "pickup_hour"])

# Add congestion (average)
congestion_df = df.groupby(["Zone_ID", "pickup_hour"])["congestion"].mean().reset_index()

merged = merged.merge(congestion_df, on=["Zone_ID", "pickup_hour"])

merged.head()

,Zone_ID,pickup_hour,demand,available_taxis,congestion
0,0,0,1647,1647,0.000852
1,0,1,956,956,0.001071
2,0,2,724,724,0.000775
3,0,3,514,514,0.000807
4,0,4,429,429,0.000883


**Prepare ML Dataset**

In [16]:
X = merged[["Zone_ID", "pickup_hour", "available_taxis", "congestion"]]
y = merged["demand"]

**Train-Test Split**

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

**Linear Regression**

In [18]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

**Decision Tree**

In [19]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=10,random_state=42)
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

**Random Forest**

In [20]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=50,random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

**Evaluation**

In [21]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate(y_true, y_pred, name, X):
    print(f"{name}")

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    n = X.shape[0]
    p = X.shape[1]
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R2:", r2)
    print("Adjusted R2:", adj_r2)
    print("-" * 30)

evaluate(y_test, lr_pred, "Linear Regression", X_test)
evaluate(y_test, dt_pred, "Decision Tree", X_test)
evaluate(y_test, rf_pred, "Random Forest", X_test)

Linear Regression
MAE: 5.423624512464661e-13
RMSE: 6.365312325374554e-13
R2: 1.0
Adjusted R2: 1.0
------------------------------
Decision Tree
MAE: 10.71875
RMSE: 22.5926333864234
R2: 0.999442076893292
Adjusted R2: 0.9994175528006894
------------------------------
Random Forest
MAE: 5.773124999999986
RMSE: 12.63428486961832
R2: 0.999825521480287
Adjusted R2: 0.9998178520948051
------------------------------


In [22]:
best_model = rf

**User Input**

In [23]:
user_lat = float(input("Enter your latitude: "))
user_lon = float(input("Enter your longitude: "))
user_available_time = float(input("Enter max wait time (minutes): "))
user_hour = int(input("Enter pickup hour (0-23): "))

Enter your latitude:  40.7851
Enter your longitude:  -73.9683
Enter max wait time (minutes): 10
Enter pickup hour (0-23): 10


**User → Zone**

In [24]:
def get_user_zone(lat, lon):
    input_df = pd.DataFrame({
        "pickup_latitude": [lat],
        "pickup_longitude": [lon]
    })
    return kmeans.predict(input_df)[0]

user_zone = get_user_zone(user_lat, user_lon)
print("User Zone:", user_zone)

User Zone: 7


**Get Taxis**

In [25]:
available_taxis = df[df["pickup_hour"] == user_hour].copy()
available_taxis = available_taxis.head(500)

**Distance**

In [26]:
available_taxis["distance_to_user"] = np.sqrt(
    (available_taxis["pickup_latitude"] - user_lat)**2 +
    (available_taxis["pickup_longitude"] - user_lon)**2
) * 111

**Time Function**

In [27]:
def Estimate_Time(distance_km, congestion):
    base_speed = 30  # km/h

    adjusted_speed = base_speed * (1 - congestion)

    if adjusted_speed < 10:
        adjusted_speed = 10

    return (distance_km / adjusted_speed) * 60

**Estimated Time**

In [28]:
available_taxis["est_time"] = available_taxis.apply(
    lambda x: Estimate_Time(x["distance_to_user"], x["congestion"]),
    axis=1
)

**Filter By User Time**

In [29]:
filtered_taxis = available_taxis[
    available_taxis["est_time"] <= user_available_time
].copy()

**Handle Empty Case**

In [30]:
print("Taxis found:", len(filtered_taxis))

if len(filtered_taxis) == 0:
    print(" No taxis available. Try increasing wait time (40 or 60).")

Taxis found: 338


**Random Forest Ranking**

In [31]:
if len(filtered_taxis) > 0:

    # Create availability feature (same as training)
    availability = df.groupby(["Zone_ID", "pickup_hour"]).size().reset_index(name="available_taxis")

    filtered_taxis = filtered_taxis.merge(
        availability,
        on=["Zone_ID", "pickup_hour"],
        how="left"
    )

    filtered_taxis["available_taxis"] = filtered_taxis["available_taxis"].fillna(0)

    # Predict demand using Random Forest
    filtered_taxis["predicted_demand"] = rf.predict(
        filtered_taxis[["Zone_ID", "pickup_hour", "available_taxis", "congestion"]]
    )

    # Create smart score (HIGH demand + LOW time)
    filtered_taxis["score"] = (
        filtered_taxis["predicted_demand"] /
        (filtered_taxis["est_time"] + 1)
    )

**Final Output**

In [32]:
def format_taxi_output(final_taxis):

    cols = [
        "pickup_latitude",
        "pickup_longitude",
        "est_time",
        "predicted_demand",
        "score"
    ]

    df_out = final_taxis[cols].head(10).copy()

    df_out["est_time"] = df_out["est_time"].round(2)
    df_out["predicted_demand"] = df_out["predicted_demand"].round(2)
    df_out["score"] = df_out["score"].round(2)

    df_out.reset_index(drop=True, inplace=True)

    # Add rank
    df_out["Rank"] = range(1, len(df_out)+1)

    # Reorder columns
    df_out = df_out[["Rank"] + cols]

    # Rename columns
    df_out.columns = [
        "Rank",
        "Latitude",
        "Longitude",
        "ETA (min)",
        "Demand",
        "Score"
    ]

    return df_out

In [33]:
final_taxis = filtered_taxis.sort_values(by="score", ascending=False)
format_taxi_output(final_taxis)

,Rank,Latitude,Longitude,ETA (min),Demand,Score
0,1,40.785122,-73.969032,0.16,1069.76,919.96
1,2,40.783092,-73.971581,0.85,1322.24,712.92
2,3,40.783901,-73.970329,0.52,1070.62,702.64
3,4,40.777233,-73.964409,1.95,2031.70,688.66
4,5,40.776379,-73.964134,2.15,2029.48,644.61
5,6,40.776176,-73.964348,2.17,2028.86,640.08
6,7,40.777588,-73.961327,2.28,2029.48,619.05
7,8,40.776520,-73.962440,2.31,2028.86,612.58
8,9,40.776974,-73.961678,2.33,2031.70,610.39
9,10,40.786186,-73.971527,0.76,1070.62,609.48


**DQN**

In [34]:
!pip install torch

In [35]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque

**Prepare Taxi Data**

In [36]:
available_taxis = df[df["pickup_hour"] == user_hour].copy()

available_taxis = available_taxis.sample(n=500, random_state=42).copy()
available_taxis.reset_index(drop=True, inplace=True)

**Calculate Distance**

In [37]:
available_taxis["distance_to_user"] = np.sqrt(
    (available_taxis["pickup_latitude"] - user_lat)**2 +
    (available_taxis["pickup_longitude"] - user_lon)**2
)

**Estimate Time**

In [38]:
def Estimate_Time(distance, congestion):
    distance_km = distance * 111
    base_speed = 30
    adjusted_speed = base_speed * (1 - congestion)

    if adjusted_speed <= 5:
        adjusted_speed = 5

    return (distance_km / adjusted_speed) * 60


available_taxis["est_time"] = available_taxis.apply(
    lambda x: Estimate_Time(x["distance_to_user"], x["congestion"]),
    axis=1
)

**Merge Availability**

In [39]:
available_taxis = available_taxis.merge(
    availability,
    on=["Zone_ID", "pickup_hour"],
    how="left"
)

if 'available_taxis' not in available_taxis.columns:
    available_taxis['available_taxis'] = 0

available_taxis["available_taxis"] = available_taxis["available_taxis"].fillna(0)

**Add Demand Prediction**

In [40]:
available_taxis["predicted_demand"] = rf.predict(
    available_taxis[["Zone_ID", "pickup_hour", "available_taxis", "congestion"]]
)

# Store original values
available_taxis["unnormalized_est_time"] = available_taxis["est_time"]
available_taxis["unnormalized_predicted_demand"] = available_taxis["predicted_demand"]

**Normalize Features**

In [41]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

features = [
    "distance_to_user",
    "est_time",
    "congestion",
    "predicted_demand"
]

available_taxis[features] = scaler.fit_transform(available_taxis[features])

**Define DQN Model**

In [42]:
import torch
import torch.nn as nn
import torch.optim as optim

class DQN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

**Define State Function**

In [43]:
def Get_State(row):
    return torch.FloatTensor([
        row["distance_to_user"],
        row["est_time"],
        row["congestion"],
        row["predicted_demand"]
    ])

**Define Reward Function**

In [44]:
def Get_Reward(row):
    reward = (
        - (row["est_time"] * 3.5)
        - (row["distance_to_user"] * 2.0)
        - (row["congestion"] * 1.5)
        + (row["predicted_demand"] * 0.5)
    )
    return reward

**Train**

In [45]:
model = DQN(input_dim=4)
optimizer = optim.Adam(model.parameters(), lr=0.0005)

epochs = 100
batch_size = 32

for epoch in range(epochs):
    total_loss = 0

    # Shuffle data each epoch (FIXED)
    shuffled = available_taxis.sample(frac=1).reset_index(drop=True)

    for i in range(0, len(shuffled), batch_size):
        batch = shuffled.iloc[i:i+batch_size]

        states = []
        rewards = []

        for _, row in batch.iterrows():
            states.append(Get_State(row))
            rewards.append(Get_Reward(row))

        states = torch.stack(states)
        rewards = torch.FloatTensor(rewards).unsqueeze(1)

        predictions = model(states)

        loss = nn.MSELoss()(predictions, rewards)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

Epoch 0, Loss: 16.5701
Epoch 10, Loss: 2.0116
Epoch 20, Loss: 0.0947
Epoch 30, Loss: 0.0220
Epoch 40, Loss: 0.0093
Epoch 50, Loss: 0.0054
Epoch 60, Loss: 0.0034
Epoch 70, Loss: 0.0026
Epoch 80, Loss: 0.0018
Epoch 90, Loss: 0.0014


**Generate DQN Scores**

In [46]:
scores = []

for i in range(len(available_taxis)):
    state = Get_State(available_taxis.iloc[i])
    score = model(state).item()
    scores.append(score)

available_taxis["DQN_score"] = scores

**Ranking + Filtering**

In [47]:
ranked_taxis = available_taxis.sort_values(by="DQN_score", ascending=False)

filtered_taxis = ranked_taxis[
    ranked_taxis["unnormalized_est_time"] <= user_available_time
].copy()

**Final Output**

In [48]:
print("Taxis found:", len(filtered_taxis))

if len(filtered_taxis) == 0:
    print("No taxis available. Increase wait time.")
else:
    min_score = filtered_taxis["DQN_score"].min()
    max_score = filtered_taxis["DQN_score"].max()

    # Normalize score safely
    filtered_taxis = filtered_taxis.copy()
    filtered_taxis["score"] = (
        (filtered_taxis["DQN_score"] - min_score) /
        (max_score - min_score + 1e-6)
    ) * 2000

    # Get top 10 taxis
    top_10 = filtered_taxis.sort_values(by="DQN_score", ascending=False).head(10).copy()

    # Create clean output dataframe (IMPORTANT FIX: .copy())
    output = top_10[[
        "pickup_latitude",
        "pickup_longitude",
        "unnormalized_est_time",
        "unnormalized_predicted_demand",
        "score"
    ]].copy()

    # Rename columns
    output.columns = [
        "Latitude",
        "Longitude",
        "ETA (min)",
        "Demand",
        "Score"
    ]

    # Safe assignment using .loc (NO WARNING)
    output.loc[:, "ETA (min)"] = output["ETA (min)"].round(2)
    output.loc[:, "Demand"] = output["Demand"].round(2)
    output.loc[:, "Score"] = output["Score"].round(2)

    # Add Rank column
    output.insert(0, "Rank", range(1, len(output) + 1))
    output.reset_index(drop=True, inplace=True)

    print("\n===== FINAL TOP 10 (DQN) =====\n")
    display(output)

Taxis found: 329

===== FINAL TOP 10 (DQN) =====



,Rank,Latitude,Longitude,ETA (min),Demand,Score
0,1,40.776958,-73.963432,2.11,2031.70,2000.00
1,2,40.779999,-73.961586,1.87,1837.94,1991.87
2,3,40.774681,-73.960129,2.94,2031.70,1960.37
3,4,40.786060,-73.968712,0.23,1071.28,1950.83
4,5,40.783485,-73.958832,2.13,1837.94,1949.44
5,6,40.776943,-73.963509,2.10,2031.70,1943.59
6,7,40.779137,-73.960480,2.18,1837.94,1937.93
7,8,40.784355,-73.973824,1.24,1321.62,1926.96
8,9,40.779968,-73.954506,3.27,1837.94,1920.41
9,10,40.787769,-73.967720,0.61,1071.28,1917.78


**Hybrid Model**

**Predict Demand (RF + Noise)**

In [49]:
base_demand = rf.predict(
    available_taxis[["Zone_ID", "pickup_hour", "available_taxis", "congestion"]]
)

noise = np.random.uniform(-300, 300, size=len(base_demand))

available_taxis["hybrid_predicted_demand"] = base_demand + noise

available_taxis["hybrid_predicted_demand"] = available_taxis["hybrid_predicted_demand"].clip(lower=0)
available_taxis["hybrid_predicted_demand"] = available_taxis["hybrid_predicted_demand"].round(2)

**RF Score**

In [50]:
available_taxis["RF_score"] = (
    available_taxis["hybrid_predicted_demand"] / # Use unnormalized demand
    (available_taxis["unnormalized_est_time"] + 1) # Use unnormalized ETA
)

**Normalize RF Score**

In [51]:
rf_min = available_taxis["RF_score"].min()
rf_max = available_taxis["RF_score"].max()

available_taxis["RF_norm"] = (
    (available_taxis["RF_score"] - rf_min) /
    (rf_max - rf_min + 1e-6)
)

**Normalize DQN Score**

In [52]:
dqn_min = available_taxis["DQN_score"].min()
dqn_max = available_taxis["DQN_score"].max()

available_taxis["DQN_norm"] = (
    (available_taxis["DQN_score"] - dqn_min) /
    (dqn_max - dqn_min + 1e-6)
)

**Hybrid Score**

In [53]:
available_taxis["Hybrid_score"] = (
    0.5 * available_taxis["RF_norm"] +
    0.5 * available_taxis["DQN_norm"]
)

available_taxis["Hybrid_score"] *= 2000

**Ranking**

In [54]:
ranked_taxis = available_taxis.sort_values(
    by="Hybrid_score",
    ascending=False
)

**Filter**

In [55]:
filtered_taxis = ranked_taxis[
    ranked_taxis["unnormalized_est_time"] <= user_available_time
].copy()

**Final Output**

In [56]:
print("Taxis found:", len(filtered_taxis))

if len(filtered_taxis) == 0:
    print("No taxis available. Increase wait time.")
else:
    top_10 = filtered_taxis.head(10).copy()

    output = top_10[[
        "pickup_latitude",
        "pickup_longitude",
        "unnormalized_est_time",
        "hybrid_predicted_demand",
        "Hybrid_score"
    ]].copy()

    output.columns = [
        "Latitude",
        "Longitude",
        "ETA (min)",
        "Demand",
        "Score"
    ]

    output.loc[:, "ETA (min)"] = output["ETA (min)"].round(2)
    output.loc[:, "Demand"] = output["Demand"].round(2)
    output.loc[:, "Score"] = output["Score"].round(2)

    output.insert(0, "Rank", range(1, len(output) + 1))
    output.reset_index(drop=True, inplace=True)

    print("\n===== FINAL TOP 10 (HYBRID MODEL) =====\n")
    display(output)

Taxis found: 329

===== FINAL TOP 10 (HYBRID MODEL) =====



,Rank,Latitude,Longitude,ETA (min),Demand,Score
0,1,40.786060,-73.968712,0.23,1043.57,1991.13
1,2,40.787769,-73.967628,0.61,1317.42,1943.77
2,3,40.778610,-73.962814,1.89,2303.49,1918.59
3,4,40.776958,-73.963432,2.11,2206.52,1838.42
4,5,40.776943,-73.963509,2.10,2181.22,1820.03
5,6,40.784355,-73.973824,1.24,1538.36,1798.34
6,7,40.786892,-73.971855,0.88,1134.98,1694.35
7,8,40.779137,-73.960480,2.18,1872.49,1682.96
8,9,40.783485,-73.958832,2.13,1792.36,1666.16
9,10,40.774662,-73.962959,2.61,2063.10,1655.61


**Map Visual**

In [57]:
!pip install folium

import folium

In [58]:

# Create Map centered at user

map_center = [user_lat, user_lon]

m = folium.Map(
    location=map_center,
    zoom_start=14
)


# Add User Marker

folium.Marker(
    location=[user_lat, user_lon],
    popup="User Location",
    icon=folium.Icon(color="red", icon="user")
).add_to(m)


# Plot Top 10 Taxis

for i, row in output.iterrows():

    # Color logic
    if i == 0:
        color = "green"   # Best taxi
    elif row["Score"] > 1900:
        color = "blue"
    else:
        color = "gray"

    folium.Marker(
        location=[row["Latitude"], row["Longitude"]],
        popup=(
            f"Rank: {i+1}<br>"
            f"ETA: {row['ETA (min)']} min<br>"
            f"Demand: {row['Demand']}<br>"
            f"Score: {row['Score']}"
        ),
        icon=folium.Icon(color=color)
    ).add_to(m)


# Draw Lines (User → Taxi)

for i, row in output.iterrows():

    folium.PolyLine(
        locations=[
            [user_lat, user_lon],
            [row["Latitude"], row["Longitude"]]
        ],
        color="green" if i == 0 else "blue",
        weight=2
    ).add_to(m)

#
# Display Map

m

**A2C**

**Import A2C Libraries**

In [62]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd

**Define A2C Network**

In [63]:
class A2CNetwork(nn.Module):
    def __init__(self, input_dim=4):
        super(A2CNetwork, self).__init__()

        # Shared feature extractor
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )

        # Actor head: outputs action probabilities (here, a score per taxi)
        self.actor = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)  # Single score output per taxi
        )

        # Critic head: estimates state value V(s)
        self.critic = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)  # State value
        )

    def forward(self, x):
        shared_out = self.shared(x)
        action_score = self.actor(shared_out)
        state_value  = self.critic(shared_out)
        return action_score, state_value

**State & Reward Functions**

In [64]:
def get_state_a2c(row):
    """Extract normalized features as a PyTorch tensor."""
    features = [
        row['distance_to_user'],
        row['est_time'],
        row['congestion'],
        row['predicted_demand']
    ]
    return torch.FloatTensor(features)

def get_reward_a2c(row):
    """
    Reward function:
    - Penalize high ETA, distance, congestion
    - Reward high predicted demand
    """
    reward = (
        -0.4 * row['est_time']
        - 0.3 * row['distance_to_user']
        - 0.2 * row['congestion']
        + 0.5 * row['predicted_demand']
    )
    return float(reward)

**Train A2C Model**

In [65]:
# ── Hyperparameters ──────────────────────────────────────────────
A2C_LR      = 0.0005
A2C_EPOCHS  = 100
GAMMA       = 0.99          # discount factor
ENTROPY_COEF = 0.01         # exploration bonus

# ── Initialise model & optimiser ────────────────────────────────
a2c_model = A2CNetwork(input_dim=4)
a2c_optimizer = optim.Adam(a2c_model.parameters(), lr=A2C_LR)

print("Training A2C Model...")
print("-" * 40)

a2c_losses = []

for epoch in range(A2C_EPOCHS):

    # Shuffle taxis each epoch
    shuffled = available_taxis.sample(frac=1, random_state=epoch).reset_index(drop=True)

    epoch_actor_loss  = 0.0
    epoch_critic_loss = 0.0
    epoch_entropy     = 0.0
    count             = 0

    states  = []
    rewards = []

    # ── Collect states & rewards for the whole epoch ─────────────
    for _, row in shuffled.iterrows():
        states.append(get_state_a2c(row))
        rewards.append(get_reward_a2c(row))

    states_tensor  = torch.stack(states)          # (N, 4)
    rewards_tensor = torch.FloatTensor(rewards)   # (N,)

    # ── Forward pass ─────────────────────────────────────────────
    action_scores, state_values = a2c_model(states_tensor)
    action_scores = action_scores.squeeze()       # (N,)
    state_values  = state_values.squeeze()        # (N,)

    # ── Compute discounted returns ────────────────────────────────
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + GAMMA * G
        returns.insert(0, G)
    returns_tensor = torch.FloatTensor(returns)

    # ── Normalise returns for stability ──────────────────────────
    if returns_tensor.std() > 1e-8:
        returns_tensor = (returns_tensor - returns_tensor.mean()) / (returns_tensor.std() + 1e-8)

    # ── Advantage = Return - Value ────────────────────────────────
    advantages = returns_tensor - state_values.detach()

    # ── Actor loss (policy gradient) ─────────────────────────────
    # Use softmax over scores → log-probs → weighted by advantage
    log_probs = torch.log_softmax(action_scores, dim=0)
    actor_loss = -(log_probs * advantages).mean()

    # ── Critic loss (value estimation) ───────────────────────────
    critic_loss = nn.MSELoss()(state_values, returns_tensor)

    # ── Entropy bonus (encourage exploration) ────────────────────
    probs   = torch.softmax(action_scores, dim=0)
    entropy = -(probs * log_probs).sum()

    # ── Total loss ────────────────────────────────────────────────
    total_loss = actor_loss + 0.5 * critic_loss - ENTROPY_COEF * entropy

    # ── Backprop ──────────────────────────────────────────────────
    a2c_optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(a2c_model.parameters(), max_norm=1.0)
    a2c_optimizer.step()

    a2c_losses.append(total_loss.item())

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1:>3}/{A2C_EPOCHS}] | "
              f"Total Loss: {total_loss.item():.4f} | "
              f"Actor: {actor_loss.item():.4f} | "
              f"Critic: {critic_loss.item():.4f} | "
              f"Entropy: {entropy.item():.4f}")

print("-" * 40)
print("A2C Training Complete!")

Training A2C Model...
----------------------------------------
Epoch [ 10/100] | Total Loss: 0.4291 | Actor: -0.0075 | Critic: 0.9976 | Entropy: 6.2145
Epoch [ 20/100] | Total Loss: 0.4013 | Actor: -0.0341 | Critic: 0.9952 | Entropy: 6.2144
Epoch [ 30/100] | Total Loss: 0.4798 | Actor: 0.0434 | Critic: 0.9971 | Entropy: 6.2142
Epoch [ 40/100] | Total Loss: 0.4107 | Actor: -0.0263 | Critic: 0.9982 | Entropy: 6.2140
Epoch [ 50/100] | Total Loss: 0.4370 | Actor: -0.0009 | Critic: 1.0001 | Entropy: 6.2139
Epoch [ 60/100] | Total Loss: 0.4908 | Actor: 0.0554 | Critic: 0.9952 | Entropy: 6.2141
Epoch [ 70/100] | Total Loss: 0.4384 | Actor: 0.0022 | Critic: 0.9967 | Entropy: 6.2142
Epoch [ 80/100] | Total Loss: 0.4256 | Actor: -0.0094 | Critic: 0.9944 | Entropy: 6.2144
Epoch [ 90/100] | Total Loss: 0.4474 | Actor: 0.0112 | Critic: 0.9967 | Entropy: 6.2145
Epoch [100/100] | Total Loss: 0.4504 | Actor: 0.0133 | Critic: 0.9984 | Entropy: 6.2145
----------------------------------------
A2C Trainin

**Generate A2C Scores**

In [66]:
a2c_model.eval()
a2c_scores = []

with torch.no_grad():
    for _, row in available_taxis.iterrows():
        state = get_state_a2c(row)
        action_score, _ = a2c_model(state.unsqueeze(0))
        a2c_scores.append(action_score.item())

available_taxis['A2C_score'] = a2c_scores
print(f"A2C scores generated for {len(a2c_scores)} taxis.")
print(f"Score range: {min(a2c_scores):.4f} → {max(a2c_scores):.4f}")

A2C scores generated for 500 taxis.
Score range: -0.0732 → 0.0297


**Rank, Filter & Display A2C Results**

In [69]:
from IPython.display import display

# ── Rank by A2C score ─────────────────────────────────────────────
ranked_taxis_a2c  = available_taxis.sort_values('A2C_score', ascending=False)
filtered_taxis_a2c = ranked_taxis_a2c[
    ranked_taxis_a2c['unnormalized_est_time'] <= user_available_time
].copy()

# ── Display ───────────────────────────────────────────────────────
if filtered_taxis_a2c.empty:
    print(f"No taxis available within {user_available_time} minutes (A2C).")
else:
    top10_a2c = filtered_taxis_a2c.head(10).copy()

    # Scale score to ~2000 range
    score_min = top10_a2c['A2C_score'].min()
    score_max = top10_a2c['A2C_score'].max()
    score_range = score_max - score_min if score_max != score_min else 1

    top10_a2c['Score'] = ((top10_a2c['A2C_score'] - score_min) / score_range * 2000).round(2)

    output_a2c = top10_a2c[[
        'pickup_latitude', 'pickup_longitude',
        'unnormalized_est_time', 'unnormalized_predicted_demand', 'Score'
    ]].copy()

    output_a2c.columns = ['Latitude', 'Longitude', 'ETA (min)', 'Demand', 'Score']

    output_a2c = output_a2c.round({
        'Latitude': 6,
        'Longitude': 6,
        'ETA (min)': 2,
        'Demand': 0,
        'Score': 2
    })

    output_a2c.insert(0, 'Rank', range(1, len(output_a2c) + 1))

    print("\n===== FINAL TOP 10 (A2C) =====\n")
    display(output_a2c)


===== FINAL TOP 10 (A2C) =====



,Rank,Latitude,Longitude,ETA (min),Demand,Score
326,1,40.774681,-73.960129,2.94,2032.0,2000.00
55,2,40.776958,-73.963432,2.11,2032.0,1631.89
284,3,40.767467,-73.953117,5.17,2032.0,1168.38
132,4,40.765015,-73.974113,4.64,2051.0,908.65
403,5,40.779968,-73.954506,3.27,1838.0,693.65
183,6,40.779999,-73.961586,1.87,1838.0,553.97
474,7,40.769493,-73.960625,3.86,2032.0,547.47
81,8,40.776943,-73.963509,2.10,2032.0,282.17
422,9,40.763386,-73.959122,5.24,2032.0,212.15
490,10,40.768940,-73.961075,3.93,2032.0,0.00


**PPO (Proximal Policy Optimization) Algorithm**

**Define PPO Network**

In [70]:
class PPONetwork(nn.Module):
    def __init__(self, input_dim=4):
        super(PPONetwork, self).__init__()

        # Shared backbone
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )

        # Actor head
        self.actor = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        # Critic head
        self.critic = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        shared_out  = self.shared(x)
        action_score = self.actor(shared_out)
        state_value  = self.critic(shared_out)
        return action_score, state_value

**State & Reward Functions for PPO**

In [71]:
def get_state_ppo(row):
    """Extract normalised features as a PyTorch tensor."""
    features = [
        row['distance_to_user'],
        row['est_time'],
        row['congestion'],
        row['predicted_demand']
    ]
    return torch.FloatTensor(features)

def get_reward_ppo(row):
    """
    Same reward shaping as A2C / DQN for fair comparison.
    """
    reward = (
        -0.4 * row['est_time']
        - 0.3 * row['distance_to_user']
        - 0.2 * row['congestion']
        + 0.5 * row['predicted_demand']
    )
    return float(reward)

**Train PPO Model**

In [72]:
# ── Hyperparameters ──────────────────────────────────────────────
PPO_LR         = 0.0003
PPO_EPOCHS     = 100
PPO_K_EPOCHS   = 4          # number of PPO update steps per epoch
PPO_CLIP_EPS   = 0.2        # clipping epsilon (core PPO parameter)
PPO_GAMMA      = 0.99
PPO_ENTROPY_C  = 0.01
PPO_CRITIC_C   = 0.5

# ── Initialise ────────────────────────────────────────────────────
ppo_model     = PPONetwork(input_dim=4)
ppo_optimizer = optim.Adam(ppo_model.parameters(), lr=PPO_LR)

print("Training PPO Model...")
print("-" * 40)

ppo_losses = []

for epoch in range(PPO_EPOCHS):

    shuffled = available_taxis.sample(frac=1, random_state=epoch).reset_index(drop=True)

    # ── Collect rollout ───────────────────────────────────────────
    states  = []
    rewards = []

    for _, row in shuffled.iterrows():
        states.append(get_state_ppo(row))
        rewards.append(get_reward_ppo(row))

    states_tensor  = torch.stack(states)           # (N, 4)
    rewards_tensor = torch.FloatTensor(rewards)    # (N,)

    # ── Compute discounted returns ────────────────────────────────
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + PPO_GAMMA * G
        returns.insert(0, G)
    returns_tensor = torch.FloatTensor(returns)

    if returns_tensor.std() > 1e-8:
        returns_tensor = (returns_tensor - returns_tensor.mean()) / \
                         (returns_tensor.std() + 1e-8)

    # ── Compute OLD log-probs (before update) ─────────────────────
    ppo_model.eval()
    with torch.no_grad():
        old_action_scores, _ = ppo_model(states_tensor)
        old_action_scores    = old_action_scores.squeeze()
        old_log_probs        = torch.log_softmax(old_action_scores, dim=0).detach()

    # ── PPO update for K epochs ───────────────────────────────────
    ppo_model.train()
    epoch_total_loss = 0.0

    for k in range(PPO_K_EPOCHS):

        action_scores, state_values = ppo_model(states_tensor)
        action_scores = action_scores.squeeze()
        state_values  = state_values.squeeze()

        # New log-probs
        new_log_probs = torch.log_softmax(action_scores, dim=0)

        # Advantage
        advantages = returns_tensor - state_values.detach()

        # ── PPO clipped surrogate objective ───────────────────────
        ratios       = torch.exp(new_log_probs - old_log_probs)
        surr1        = ratios * advantages
        surr2        = torch.clamp(ratios, 1 - PPO_CLIP_EPS,
                                           1 + PPO_CLIP_EPS) * advantages
        actor_loss   = -torch.min(surr1, surr2).mean()

        # ── Critic loss ───────────────────────────────────────────
        critic_loss  = nn.MSELoss()(state_values, returns_tensor)

        # ── Entropy bonus ─────────────────────────────────────────
        probs        = torch.softmax(action_scores, dim=0)
        entropy      = -(probs * new_log_probs).sum()

        # ── Total loss ────────────────────────────────────────────
        total_loss = actor_loss + PPO_CRITIC_C * critic_loss - PPO_ENTROPY_C * entropy

        ppo_optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(ppo_model.parameters(), max_norm=1.0)
        ppo_optimizer.step()

        epoch_total_loss += total_loss.item()

    avg_loss = epoch_total_loss / PPO_K_EPOCHS
    ppo_losses.append(avg_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1:>3}/{PPO_EPOCHS}] | "
              f"Avg Loss: {avg_loss:.4f} | "
              f"Actor: {actor_loss.item():.4f} | "
              f"Critic: {critic_loss.item():.4f} | "
              f"Entropy: {entropy.item():.4f}")

print("-" * 40)
print("PPO Training Complete!")

Training PPO Model...
----------------------------------------
Epoch [ 10/100] | Avg Loss: 0.4427 | Actor: 0.0059 | Critic: 0.9973 | Entropy: 6.2143
Epoch [ 20/100] | Avg Loss: 0.4331 | Actor: -0.0006 | Critic: 0.9927 | Entropy: 6.2126
Epoch [ 30/100] | Avg Loss: 0.4327 | Actor: -0.0026 | Critic: 0.9969 | Entropy: 6.2136
Epoch [ 40/100] | Avg Loss: 0.4413 | Actor: 0.0043 | Critic: 0.9988 | Entropy: 6.2130
Epoch [ 50/100] | Avg Loss: 0.4441 | Actor: 0.0042 | Critic: 1.0007 | Entropy: 6.2122
Epoch [ 60/100] | Avg Loss: 0.4362 | Actor: -0.0005 | Critic: 0.9963 | Entropy: 6.2145
Epoch [ 70/100] | Avg Loss: 0.4350 | Actor: -0.0007 | Critic: 0.9966 | Entropy: 6.2144
Epoch [ 80/100] | Avg Loss: 0.4375 | Actor: 0.0002 | Critic: 0.9963 | Entropy: 6.2145
Epoch [ 90/100] | Avg Loss: 0.4371 | Actor: 0.0003 | Critic: 0.9968 | Entropy: 6.2145
Epoch [100/100] | Avg Loss: 0.4352 | Actor: -0.0006 | Critic: 0.9981 | Entropy: 6.2145
----------------------------------------
PPO Training Complete!


**Generate PPO Scores**

In [73]:
ppo_model.eval()
ppo_scores = []

with torch.no_grad():
    for _, row in available_taxis.iterrows():
        state = get_state_ppo(row)
        action_score, _ = ppo_model(state.unsqueeze(0))
        ppo_scores.append(action_score.item())

available_taxis['PPO_score'] = ppo_scores
print(f"PPO scores generated for {len(ppo_scores)} taxis.")
print(f"Score range: {min(ppo_scores):.4f} → {max(ppo_scores):.4f}")

PPO scores generated for 500 taxis.
Score range: -0.3099 → -0.2210


**Rank, Filter & Display PPO Results**

In [75]:
from IPython.display import display

# ── Rank by PPO score ─────────────────────────────────────────────
ranked_taxis_ppo   = available_taxis.sort_values('PPO_score', ascending=False)
filtered_taxis_ppo = ranked_taxis_ppo[
    ranked_taxis_ppo['unnormalized_est_time'] <= user_available_time
].copy()

# ── Display ───────────────────────────────────────────────────────
if filtered_taxis_ppo.empty:
    print(f"No taxis available within {user_available_time} minutes (PPO).")
else:
    top10_ppo = filtered_taxis_ppo.head(10).copy()

    score_min = top10_ppo['PPO_score'].min()
    score_max = top10_ppo['PPO_score'].max()
    score_range = score_max - score_min if score_max != score_min else 1

    top10_ppo['Score'] = ((top10_ppo['PPO_score'] - score_min) / score_range * 2000).round(2)

    output_ppo = top10_ppo[[
        'pickup_latitude', 'pickup_longitude',
        'unnormalized_est_time', 'unnormalized_predicted_demand', 'Score'
    ]].copy()

    output_ppo.columns = ['Latitude', 'Longitude', 'ETA (min)', 'Demand', 'Score']

    output_ppo = output_ppo.round({
        'Latitude': 6,
        'Longitude': 6,
        'ETA (min)': 2,
        'Demand': 0,
        'Score': 2
    })

    output_ppo.insert(0, 'Rank', range(1, len(output_ppo) + 1))

    # ✅ Table display
    display(output_ppo)

,Rank,Latitude,Longitude,ETA (min),Demand,Score
326,1,40.774681,-73.960129,2.94,2032.0,2000.00
284,2,40.767467,-73.953117,5.17,2032.0,1917.84
132,3,40.765015,-73.974113,4.64,2051.0,1523.17
55,4,40.776958,-73.963432,2.11,2032.0,1050.33
422,5,40.763386,-73.959122,5.24,2032.0,795.91
474,6,40.769493,-73.960625,3.86,2032.0,693.62
369,7,40.758263,-73.966164,5.98,2051.0,329.44
403,8,40.779968,-73.954506,3.27,1838.0,270.74
437,9,40.758671,-73.977173,6.19,2051.0,208.18
490,10,40.768940,-73.961075,3.93,2032.0,0.00
